# Finetune Qwen3-4B-Instruct-2507 to create sturctured clinical notes from patient-doctor dialogue


Steps in this notebook: 
1. Import libraries required for finetuning
2. Set variables
3. Define preprocessing function (tokenization)
4. Load tokenizer and model
5. Apply LoRA
6. Load and set up the data
7. Set up the HuggingFace Trainer
8. **Start the finetuning process**
9. Monitor GPU utilization from terminal while the training is running
10. Save the model
11. BONUS: Check SLURM job ID
12. Restart kernel

# ⚠️ First Step: Run All Cells ⚠️

Before proceeding, start the notebook execution.  
From the menu in the top-left corner, select: **Run → Run All Cells**  
The notebook will run in the background while we go through this notebook step by step.   

⚠️ **Run one notebook at a time.** Running multiple notebooks simultaneously may cause memory issues and kernel failures.

*A kernel is the process that executes the notebook's code and stores variables, data, and model state in memory. If available memory is exhausted, the kernel may become unresponsive or restart.*

# 1. Import libraries required for finetuning

Before we can train a model, we need to load the tools we'll use. Think of these as the specialized software packages that handle different parts of the process:

- **torch** — PyTorch, the core deep learning framework that runs computations on the GPU
- **transformers** — Hugging Face's library for loading and working with pretrained language models
- **peft** — Enables *LoRA*, a technique that makes finetuning much cheaper by only training a small fraction of the model's parameters
- **datasets** — Handles loading and processing our training data efficiently
- **pandas** — General-purpose data manipulation, used here to load our JSON data file

These are already installed to the chosen module.


In [ ]:
import argparse
import os
import sys
import time
import torch
import mlflow
import pandas as pd

from datasets import Dataset 

from pathlib import Path
from datasets import load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    AutoProcessor
)
from functools import partial

import warnings
warnings.filterwarnings("ignore")

## 2. Set variables that will be used e.g. in model loading, finetuning and loading data

- **`SLURM_JOB_ACCOUNT` and `SLURM_JOB_USER`** — read automatically from the LUMI environment, used to build file paths
- **`input_model`** — the pretrained model we start from: [Qwen/Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). Changing this to another Hugging Face model ID will finetune that model instead
- **`json_file`** — path to your training data file
- **`batch_size`** — how many examples the model processes at once. Higher is faster but uses more GPU memory — `2` is a safe starting point for a single GPU
- **`max_tokens`** — maximum length of a single training example in tokens (roughly words). Examples longer than this are cut off at the end
- **`cache_dir`** — where Hugging Face downloads and caches model weights. Pointing this to scratch space prevents the model files from filling up your home directory quota
- **`num_workers`** — number of CPU threads used for data loading, read from the SLURM allocation so it scales automatically with your job
- **`output_model_dir`** — where the finetuned model checkpoint will be saved
- **`merged_output_dir`** — where the final merged model is saved after LoRA weights are merged back into the base model, ready for inference

In [ ]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT")
USER = os.getenv("SLURM_JOB_USER")

print(f"Billing project:  {SLURM_JOB_ACCOUNT}")
print(f"Username:         {USER}")

In [ ]:
input_model = "Qwen/Qwen3-4B-Instruct-2507"
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
model_output_name = f"{input_model}_finetuned"
json_file = f"/scratch/{SLURM_JOB_ACCOUNT}/data/structured_notes.json"
batch_size = 2
cache_dir = f"/scratch/{SLURM_JOB_ACCOUNT}/hf-cache/hub"
max_tokens = 2048
num_workers = int(os.getenv("SLURM_CPUS_PER_TASK"))

output_model_dir = os.path.join(output_path, model_output_name)
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

print(f"Model will be saved to: {merged_output_dir}")

## 3. Define preprocessing function (tokenization)

This cell defines the `system_prompt` and a `preprocess` function that converts raw conversations into tokenized format the model can learn from.

Models don't read text directly — they read **tokens**, numbers representing words or parts of words. The function builds each training example as a three-part conversation, tokenizes it, then masks the prompt portion in the labels with `-100` so the model is only trained to predict the assistant's response, not repeat the question back.

The three conversation roles are:
- `system` — instructions that tell the model what its job is (here: converting doctor-patient dialogue to a clinical note)
- `user` — the raw doctor-patient conversation (the input)
- `assistant` — the correct structured note we want the model to produce (the expected output)

In [ ]:
system_prompt = """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""

def preprocess(examples, tokenizer, system_prompt, max_tokens=2048):
    """
    Convert chat examples into tokenized causal-LM training examples.

    Safeguards:
    - Skip samples where the assistant response is completely truncated.
    - Ensure labels and input_ids always have identical lengths.
    - Ensure at least one non-masked label token exists.
    """

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    skipped_no_response = 0
    skipped_empty_labels = 0

    for conversation, structured_note in zip(
        examples["conversation"],
        examples["structured_note"],
    ):

        # Build chat messages
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": conversation},
            {"role": "assistant", "content": structured_note},
        ]

        # Apply chat template — tokenize full conversation
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        prompt_text = tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=False,
            add_generation_prompt=True,
        )

        full_enc = tokenizer(
            full_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
            add_special_tokens=False,
        )

        prompt_enc = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
            add_special_tokens=False,
        )

        input_ids = full_enc["input_ids"]
        attention_mask = full_enc["attention_mask"]

        # Safety: prompt length cannot exceed actual sequence length
        prompt_len = min(len(prompt_enc["input_ids"]), len(input_ids))

        # Assistant response completely truncated
        if prompt_len >= len(input_ids):
            skipped_no_response += 1
            continue

        labels = [-100] * prompt_len + input_ids[prompt_len:]

        # Safety check
        if len(labels) != len(input_ids):
            raise ValueError(
                f"Label length mismatch: labels={len(labels)}, "
                f"input_ids={len(input_ids)}"
            )

        valid_label_count = sum(label != -100 for label in labels)

        if valid_label_count == 0:
            skipped_empty_labels += 1
            continue

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

print(f"Used system_prompt:\n\n{system_prompt}")

## 4. Load tokenizer & models

These cells detect the GPU, load the pretrained model and tokenizer, then wrap the model with LoRA.

**GPU detection** — checks whether a GPU is available. Training on CPU would be impractically slow — a step that takes 1 second on GPU can take 50–100 seconds on CPU.

**Tokenizer** — must match the model exactly as each model has its own vocabulary. The `pad_token` fix ensures the tokenizer can handle batches of different-length examples.

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(input_model, use_fast=True, cache_dir=cache_dir)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded")
print(f"GPU memory before model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    input_model,
    torch_dtype=torch.bfloat16,
    device_map=device,
    cache_dir=cache_dir
)

print(f"GPU memory after model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")

## 5. Apply LoRA (Low-Rank Adaptation)
The amount of trainable parameters depends on the model architecture, the `r` value, and which layers LoRA is applied to. With our configuration on this 4B model, only ~0.89% of parameters are trained (~38M out of 4.3B total).

Key settings:
- `r=16` — the rank, controls the size of the adapters. Higher means more learning capacity but more memory. Common values are 8, 16, and 32
- `lora_alpha=8` — a scaling factor for the adapter outputs, typically set to `r/2` or equal to `r`
- `lora_dropout=0.05` — randomly disables 5% of adapter connections during training to prevent the model from memorising the training data
- `target_modules="all-linear"` — applies LoRA to all linear layers in the model. You could instead target only specific layers such as attention layers, which would reduce the trainable parameter count further

In [ ]:
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Load and set up the data

This cell loads the training data and splits it into two subsets: one for training and one for validation.

The full dataset contains 30 000 doctor-patient conversation and structured note pairs, but here we use only 2 000 examples — enough for a meaningful finetuning run within the time and memory limits of a single-GPU notebook session.

The dataset has several columns but we only use two:
- **`conversation`** — the raw doctor-patient dialogue, used as the model input
- **`structured_note`** — the target clinical note the model learns to produce

The data is split 95/5 into training and validation sets. The validation set is held back during training to measure how well the model generalises to unseen examples. `seed=42` ensures the split is the same every time you run the cell, which is important for reproducibility.

In [ ]:
## DATA
df = pd.read_json(json_file)[:2000]
dataset = Dataset.from_pandas(df, preserve_index=False)
split = dataset.train_test_split(test_size=0.05, seed=42)

raw_train = split["train"]
raw_val = split["test"]

print(f"  Train size: {len(raw_train)}")
print(f"  Val size:   {len(raw_val)}")


We will also quickly inspect at some of the examples of the data. `conversation` and `structured_notes` will be used as the finetuning material.

In [ ]:
dataset_df = pd.DataFrame(raw_val)
df.head(1)

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.html.use_mathjax', False)
df[['conversation', 'structured_note']].head(1).style.set_properties(**{'white-space': 'pre-wrap', 'width': '500px'})

## Tokenize datasets

In [ ]:
preprocess_fn = partial(preprocess, system_prompt=system_prompt, tokenizer=tokenizer, max_tokens=max_tokens)

In [ ]:
tokenized_train = raw_train.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_train.column_names,
    num_proc=num_workers,
)

In [ ]:
tokenized_val = raw_val.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_val.column_names,
    num_proc=num_workers,
)

## 7. Setting up the HuggingFace Trainer with required arguments for finetuning

These cells configure the training behaviour, set up data batching, and initialize the Trainer that orchestrates everything.

**`TrainingArguments`** defines how training behaves: learning rate, how many epochs to train, how often to evaluate and save checkpoints, and what precision to use. The most important ones are set in the config cell — the rest are sensible defaults for LoRA finetuning.

**`DataCollatorForSeq2Seq`** pads examples to the same length so they can be processed as a batch, aligned to multiples of 8 for GPU efficiency.

**`Trainer`** ties everything together — model, arguments, datasets, tokenizer, and collator. The next cell starts training with a single call to `trainer.train()`.

In [ ]:
training_args = TrainingArguments(
    disable_tqdm=False,
    output_dir=output_model_dir,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    load_best_model_at_end=True,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size*2,
    dataloader_num_workers=num_workers,
    ddp_find_unused_parameters=True,
    dataloader_pin_memory=True,
    #save_safetensors=True,
    metric_for_best_model="eval_loss",
    eval_strategy="steps",
    eval_steps=150,
    num_train_epochs=1,
    report_to=["mlflow"],
    logging_steps=50,
    logging_strategy="steps",
    run_name=f"{model_output_name}_{os.environ.get('SLURM_JOB_ID', 'local')}",
)

print(f"Training for {training_args.num_train_epochs} epoch(s) | Eval every {training_args.eval_steps} steps")

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

print("Data collator ready")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    #tokenizer=tokenizer,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(f"Trainer ready")

# 8. Start the finetuning process

This cell starts the training. During training you will see a table printed every 150 steps showing:

- **Step** — how many training batches have been processed so far
- **Training Loss** — how wrong the model's predictions are on the training data, should decrease over time
- **Validation Loss** — how wrong the predictions are on the held-out validation examples the model has never seen. If this stops improving while training loss keeps dropping, the model is starting to memorise the training data rather than learning general patterns

With 2000 examples and `batch_size=2` on a single GPU expect roughly 20 minutes.

> **Note:** You may see an MLflow message appearing during training — this is expected and can be ignored. `report_to=["mlflow"]` is set in the `TrainigArguments` purely to prevent an automatic connection to an external logging service (Weights & Biases) that would otherwise cause errors in this environment.

In [ ]:
start_train = time.time()

trainer.train()

stop_train = time.time()

In [ ]:
elapsed = stop_train - start_train
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"Training took: {hours}h {minutes}m {seconds}s")

## 9. Monitor GPU utilization from terminal while the training is running

While training is running, open a **new terminal inside the Jupyter App** and run:

```bash
hostname    # confirm which compute node you are on
rocm-smi    # show current GPU memory usage and utilization
```

`rocm-smi` is the AMD equivalent of `nvidia-smi` — it shows GPU memory usage, temperature, and compute utilization. Run it manually a few times during training to see the GPU being utilized. You should see one GPU active with memory usage climbing once training starts.

If you submitted via `sbatch` instead, monitoring works differently — you would need to find the job ID and open an interactive session on the compute node manually. See the [LUMI AI Guide](https://github.com/Lumi-supercomputer/LUMI-AI-Guide/tree/main/06-monitoring-and-profiling) for step-by-step instructions.

## 10. Merge LoRA adapters and save the model

This is the final step — merging the trained LoRA adapters back into the base model and saving everything to disk.

During training, LoRA kept the original model weights frozen and only updated the small adapter layers. `merge_and_unload()` combines these adapters back into the base model weights, producing a single standalone model that can be loaded and run without any dependency on the PEFT library.

The merged model, tokenizer, and processor are all saved to `merged_output_dir` defined in the config cell. This folder is used as input in the next inference notebook.

In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    merged_output_dir,
    safe_serialization=False
)

tokenizer.save_pretrained(merged_output_dir)

## 11. Check SLURM Job ID

In a typical HPC workflow you submit a job from the terminal with `sbatch my_script.sh` and — if there are no errors in the script setup — get a job ID back immediately. The job then runs unattended in the background.

This notebook works differently. When you launched the Jupyter App on LUMI and selected your resources (GPUs, memory, time limit), the system automatically submitted a SLURM job behind the scenes to reserve those resources. **The Jupyter session itself is the SLURM job** — all code you run in this notebook executes within that already-allocated job.

This means:
- The cell below prints the job ID of your current Jupyter session
- When the time limit expires, the session ends and any running cells are interrupted
- There is no separate SLURM script needed for this notebook — unlike the multi-GPU `finetune.py` which is submitted via `sbatch` and runs without a Jupyter session

In [ ]:
# Check Slurm job ID
print(os.environ.get("SLURM_JOB_ID", "Not running inside a SLURM job"))

# 12. Restart kernel before continuing

Before moving to the inference notebook, restart the kernel to free the GPU memory occupied by the finetuned model and trainer. Without this, the next notebook may run out of memory when trying to load a model.

**Kernel → Restart Kernel** in the Jupyter menu, or run the cell below.

In [ ]:
from IPython import get_ipython

get_ipython().kernel.do_shutdown(restart=True)

## Move on to next notebook
